# Compact VITS-Style Text-to-Speech from Scratch

This notebook builds a small, single-speaker text-to-speech system using the main ideas behind VITS:

1. A **text encoder** predicts a text-conditioned latent prior.
2. A **posterior encoder** converts the target spectrogram into a latent sequence.
3. A **normalizing flow** connects the posterior and prior latent spaces.
4. **Monotonic Alignment Search (MAS)** aligns text tokens with audio frames.
5. A **duration predictor** learns how long each token should last.
6. A waveform **decoder** upsamples latent frames into audio samples.

> **Scope:** this is an educational, non-adversarial VITS-style implementation. It trains with mel, multi-resolution STFT, duration, and KL losses. The discriminator, adversarial loss, and feature-matching loss used by the full VITS model are intentionally omitted.


In [ ]:
import math
from pathlib import Path

# gan-group start: for evaluation plotting
import matplotlib.pyplot as plt
# gan-group end

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


## 1. Configuration and hardware

The audio settings make one spectrogram frame correspond to `HOP_LENGTH = 256` waveform samples. A training segment therefore contains 32 latent frames and produces 8,192 audio samples.

The hardware check is also important on Kaggle. When **GPU T4 ×2** is enabled, `torch.cuda.device_count()` should report `2`. The trainer later uses `nn.DataParallel` to divide each global batch across both visible GPUs.


In [ ]:
# Dataset paths on Kaggle
DATA_DIR = Path("/kaggle/input/datasets/mathurinache/the-lj-speech-dataset/LJSpeech-1.1")
WAV_DIR = DATA_DIR / "wavs"
METADATA_PATH = DATA_DIR / "metadata.csv"

# Audio representation
SAMPLE_RATE = 22_050
N_FFT = 1_024
HOP_LENGTH = 256
WIN_LENGTH = 1_024

# The decoder sees 32 spectrogram frames at a time:
# 32 frames × 256 samples per frame = 8,192 waveform samples.
SEGMENT_SAMPLES = 8_192
SEGMENT_FRAMES = SEGMENT_SAMPLES // HOP_LENGTH

# This is the global batch size. With two GPUs, DataParallel splits it
# approximately evenly, so batch_size=8 gives four samples to each T4.
BATCH_SIZE = 8
NUM_WORKERS = 4

# Show the accelerator configuration before training.
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()

print(f"PyTorch version: {torch.__version__}")
print(f"Primary device: {DEVICE}")
print(f"Visible CUDA GPUs: {NUM_GPUS}")

for gpu_index in range(NUM_GPUS):
    print(f"  GPU {gpu_index}: {torch.cuda.get_device_name(gpu_index)}")


## 2. Load the LJSpeech metadata

Each LJSpeech row contains a file ID, the original transcription, and a normalized transcription. Only the file ID and normalized text are required for this TTS pipeline.


In [ ]:
metadata_df = pd.read_csv(
    METADATA_PATH,
    sep="|",
    header=None,
    names=[
        "file_name",
        "transcription",
        "normalized_transcription"
    ]
)

# TTS only needs the audio filename and its text
metadata_df = metadata_df[
    ["file_name", "normalized_transcription"]
].dropna().reset_index(drop=True)

metadata_df.head()

## 3. Convert text into token IDs

The model cannot process characters directly, so every supported character is assigned an integer ID. ID `0` is reserved for padding, which allows text sequences of different lengths to share one batch.

This character-level tokenizer is intentionally simple. Unsupported characters are ignored rather than mapped to an unknown token.


In [ ]:
PAD_SYMBOL = "_"
PUNCTUATION = ";:,.!?¡¿—…\"«»“”ü ()-'[]"
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

SYMBOLS = [PAD_SYMBOL] + list(PUNCTUATION) + list(LETTERS)

symbol_to_id = {
    symbol: index
    for index, symbol in enumerate(SYMBOLS)
}

PAD_ID = symbol_to_id[PAD_SYMBOL]

def encode_text(text):
    """
    Convert a string into a sequence of integer token IDs.

    Example:
        "Hi" -> tensor([token_id_for_H, token_id_for_i])
    """
    token_ids = [
        symbol_to_id[character]
        for character in text
        if character in symbol_to_id
    ]

    return torch.tensor(token_ids, dtype=torch.long)

## 4. Audio preprocessing and dataset

The waveform is converted into a magnitude spectrogram with Short-Time Fourier Transform (STFT). With `N_FFT = 1024`, the one-sided spectrogram contains `1024 / 2 + 1 = 513` frequency bins.

`LJSpeechDataset` returns three items for each utterance: encoded text, spectrogram, and waveform. The spectrogram is used by the posterior encoder, while the original waveform supplies the reconstruction target.


In [ ]:
hann_window = torch.hann_window(WIN_LENGTH)

def waveform_to_spectrogram(waveform):
    """
    Convert a waveform from shape [1, audio_length]
    into a magnitude spectrogram with shape
    [frequency_bins, time_frames].
    """
    padding = (N_FFT - HOP_LENGTH) // 2

    waveform = F.pad(
        waveform,
        (padding, padding),
        mode="reflect"
    )

    complex_spectrogram = torch.stft(
        waveform,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=hann_window,
        center=False,
        normalized=False,
        onesided=True,
        return_complex=True
    )

    # Convert the complex STFT into magnitude
    magnitude_spectrogram = complex_spectrogram.abs()

    # [1, frequency_bins, time_frames]
    # -> [frequency_bins, time_frames]
    return magnitude_spectrogram.squeeze(0)

In [ ]:
class LJSpeechDataset(Dataset):
    def __init__(self, metadata, wav_dir):
        self.metadata = metadata
        self.wav_dir = Path(wav_dir)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]

        # 1. Load and encode the text
        text = encode_text(row["normalized_transcription"])

        # 2. Load the waveform
        audio_path = self.wav_dir / f"{row['file_name']}.wav"
        waveform, sample_rate = torchaudio.load(audio_path)

        if sample_rate != SAMPLE_RATE:
            raise ValueError(
                f"Expected sample rate {SAMPLE_RATE}, "
                f"but received {sample_rate} for {audio_path.name}"
            )

        # LJSpeech is mono, so waveform has shape [1, audio_length]
        if waveform.size(0) != 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # 3. Convert the waveform into a spectrogram
        spectrogram = waveform_to_spectrogram(waveform)

        return text, spectrogram, waveform

### Padding variable-length samples

Utterances and transcriptions have different lengths. The collate function records every original length, pads each batch to its longest item, and returns the lengths so later masks can distinguish real values from padding.


In [ ]:
def collate_tts_batch(batch):
    """
    Combine samples of different lengths into one padded batch.
    """
    texts, spectrograms, waveforms = zip(*batch)

    # Record the original lengths before padding
    text_lengths = torch.tensor(
        [text.size(0) for text in texts],
        dtype=torch.long
    )

    spec_lengths = torch.tensor(
        [spectrogram.size(1) for spectrogram in spectrograms],
        dtype=torch.long
    )

    wav_lengths = torch.tensor(
        [waveform.size(1) for waveform in waveforms],
        dtype=torch.long
    )

    # Text:
    # List of [text_length] -> [batch, max_text_length]
    text_padded = pad_sequence(
        texts,
        batch_first=True,
        padding_value=PAD_ID
    )

    # Spectrogram:
    # [frequency, time] -> [time, frequency]
    # so pad_sequence can pad the time dimension
    spec_padded = pad_sequence(
        [
            spectrogram.transpose(0, 1)
            for spectrogram in spectrograms
        ],
        batch_first=True,
        padding_value=0.0
    )

    # [batch, time, frequency]
    # -> [batch, frequency, time]
    spec_padded = spec_padded.transpose(1, 2)

    # Waveform:
    # [1, time] -> [time]
    wav_padded = pad_sequence(
        [
            waveform.squeeze(0)
            for waveform in waveforms
        ],
        batch_first=True,
        padding_value=0.0
    )

    # [batch, time] -> [batch, 1, time]
    wav_padded = wav_padded.unsqueeze(1)

    return (
        text_padded,
        text_lengths,
        spec_padded,
        spec_lengths,
        wav_padded,
        wav_lengths
    )

In [ ]:
train_dataset = LJSpeechDataset(
    metadata=metadata_df,
    wav_dir=WAV_DIR,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_tts_batch,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)

batch = next(iter(train_loader))

(
    text_padded,
    text_lengths,
    spec_padded,
    spec_lengths,
    wav_padded,
    wav_lengths
) = batch

print("Text padded:       ", text_padded.shape)
print("Text lengths:      ", text_lengths.shape)
print("Spectrogram padded:", spec_padded.shape)
print("Spectrogram lengths:", spec_lengths.shape)
print("Waveform padded:   ", wav_padded.shape)
print("Waveform lengths:  ", wav_lengths.shape)

## 5. Sequence masks

A mask marks valid positions with `True` and padded positions with `False`. Every encoder and loss uses these masks so padding does not affect representations, alignments, or loss values.


In [ ]:
def create_sequence_mask(lengths, max_length):
    """
    Create a mask for sequences with different lengths.

    lengths:
        [3, 5]

    result:
        [[True, True, True, False, False],
         [True, True, True, True,  True ]]
    """
    positions = torch.arange(
        max_length,
        device=lengths.device
    )

    return positions.unsqueeze(0) < lengths.unsqueeze(1)

## 6. Text encoder and latent prior

The text encoder combines character and position embeddings, then applies Transformer encoder layers. For every text token it returns:

- contextual text features for the duration predictor;
- a prior mean $\mu_p$;
- a prior log standard deviation $\log \sigma_p$;
- a text mask.

These statistics describe a **text-conditioned Gaussian prior**. MAS later expands the token-level prior to spectrogram-frame length.


In [ ]:
class TextEncoder(nn.Module):
    """
    Convert text token IDs into:

    1. Encoded text features
    2. Prior mean
    3. Prior log-scale
    4. Text mask
    """

    def __init__(
        self,
        vocab_size,
        hidden_channels=192,
        latent_channels=192,
        feedforward_channels=768,
        num_heads=2,
        num_layers=6,
        dropout=0.1,
        max_text_length=512,
        padding_id=0
    ):
        super().__init__()

        self.hidden_channels = hidden_channels
        self.max_text_length = max_text_length

        # Convert token IDs into vectors
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=hidden_channels,
            padding_idx=padding_id
        )

        # Give the Transformer information about token positions
        self.position_embedding = nn.Embedding(
            num_embeddings=max_text_length,
            embedding_dim=hidden_channels
        )

        # One Transformer block:
        # self-attention + feed-forward network
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=hidden_channels,
            nhead=num_heads,
            dim_feedforward=feedforward_channels,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        # Stack multiple Transformer blocks
        self.transformer = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=num_layers
        )

        # Produce mean and log-scale together
        self.statistics_projection = nn.Linear(
            hidden_channels,
            latent_channels * 2
        )

    def forward(self, token_ids, text_lengths):
        """
        token_ids:
            [batch, text_length]

        text_lengths:
            [batch]
        """
        batch_size, text_length = token_ids.shape

        if text_length > self.max_text_length:
            raise ValueError(
                f"Text length {text_length} exceeds "
                f"max_text_length={self.max_text_length}"
            )

        # --------------------------------------------------
        # 1. Create text mask
        # --------------------------------------------------

        valid_mask = create_sequence_mask(
            text_lengths,
            max_length=text_length
        )  # [batch, text_length]

        # PyTorch Transformer uses True to mean:
        # "ignore this padding position"
        padding_mask = ~valid_mask

        # --------------------------------------------------
        # 2. Create token and position embeddings
        # --------------------------------------------------

        token_features = self.token_embedding(token_ids)

        position_ids = torch.arange(
            text_length,
            device=token_ids.device
        ).unsqueeze(0)

        position_features = self.position_embedding(position_ids)

        # [batch, text_length, hidden_channels]
        encoded_text = token_features + position_features

        # --------------------------------------------------
        # 3. Apply the Transformer
        # --------------------------------------------------

        encoded_text = self.transformer(
            encoded_text,
            src_key_padding_mask=padding_mask
        )

        # Set padding positions to zero
        float_mask = valid_mask.unsqueeze(-1).to(encoded_text.dtype)
        encoded_text = encoded_text * float_mask

        # --------------------------------------------------
        # 4. Predict prior distribution statistics
        # --------------------------------------------------

        statistics = self.statistics_projection(encoded_text)

        prior_mean, prior_log_scale = statistics.chunk(
            chunks=2,
            dim=-1
        )

        # VITS usually expects [batch, channels, time]
        encoded_text = encoded_text.transpose(1, 2)
        prior_mean = prior_mean.transpose(1, 2)
        prior_log_scale = prior_log_scale.transpose(1, 2)

        # [batch, 1, text_length]
        text_mask = valid_mask.unsqueeze(1).to(encoded_text.dtype)

        return (
            encoded_text,
            prior_mean,
            prior_log_scale,
            text_mask
        )


In [ ]:
text_encoder = TextEncoder(
    vocab_size=len(SYMBOLS),
    padding_id=PAD_ID
)
text_tokens = batch[0]
text_lengths = batch[1]

encoded_text, prior_mean, prior_log_scale, text_mask = text_encoder(
    text_tokens,
    text_lengths
)

print("Encoded text:   ", encoded_text.shape)
print("Prior mean:     ", prior_mean.shape)
print("Prior log-scale:", prior_log_scale.shape)
print("Text mask:      ", text_mask.shape)

## 7. Posterior encoder

During training, the posterior encoder observes the full target spectrogram and predicts $q(z\mid y)$. Dilated convolutions capture local audio context without changing the number of time frames.

The latent sample is generated with the reparameterization trick:

$$z = \mu_q + \epsilon \exp(\log \sigma_q), \qquad \epsilon \sim \mathcal{N}(0, I).$$


In [ ]:
class PosteriorEncoder(nn.Module):
    """
    Convert a spectrogram into a posterior latent variable.

    Input
    -----
    spectrogram:
        [batch, spec_channels, time]

    spectrogram_lengths:
        [batch]

    Output
    ------
    z:
        Sampled latent representation

    posterior_mean:
        Mean of the posterior distribution

    posterior_log_scale:
        Log standard deviation of the posterior distribution

    spectrogram_mask:
        Mask for valid spectrogram frames
    """

    def __init__(
        self,
        spec_channels,
        latent_channels=192,
        hidden_channels=192,
        dropout=0.1
    ):
        super().__init__()
        # Convert spectrogram channels into hidden channels
        self.input_projection = nn.Conv1d(spec_channels, hidden_channels, kernel_size=1)

        # Dilated convolution layers
        # Padding is selected so that the time length
        # remains unchanged.
        self.conv1 = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=5,
            dilation=1,
            padding=2
        )

        self.conv2 = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=5,
            dilation=2,
            padding=4
        )

        self.conv3 = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=5,
            dilation=4,
            padding=8
        )

        self.conv4 = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=5,
            dilation=8,
            padding=16
        )

        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)

        # Separate layers make the two outputs easier to understand
        self.mean_projection = nn.Conv1d(
            hidden_channels,
            latent_channels,
            kernel_size=1
        )

        self.log_scale_projection = nn.Conv1d(
            hidden_channels,
            latent_channels,
            kernel_size=1
        )

    def forward(self, spectrogram, spectrogram_lengths):
        time_length = spectrogram.size(2)

        # [batch, time]
        valid_mask = create_sequence_mask(
            spectrogram_lengths,
            max_length=time_length
        )
        
        # [batch, 1, time]
        spectrogram_mask = valid_mask.unsqueeze(1).to(
            spectrogram.dtype
        )

        # --------------------------------------------------
        # CNN feature extraction
        # --------------------------------------------------
        hidden = self.input_projection(spectrogram)
        hidden = self.activation(hidden) * spectrogram_mask
        
        hidden = self.conv1(hidden)
        hidden = self.dropout(self.activation(hidden))
        hidden = hidden * spectrogram_mask

        hidden = self.conv2(hidden)
        hidden = self.dropout(self.activation(hidden))
        hidden = hidden * spectrogram_mask

        hidden = self.conv3(hidden)
        hidden = self.dropout(self.activation(hidden))
        hidden = hidden * spectrogram_mask
        
        hidden = self.conv4(hidden)
        hidden = self.dropout(self.activation(hidden))
        hidden = hidden * spectrogram_mask

        # --------------------------------------------------
        # Posterior distribution
        # --------------------------------------------------

        posterior_mean = (
            self.mean_projection(hidden)
            * spectrogram_mask
        )
        
        posterior_log_scale = (
            self.log_scale_projection(hidden)
            * spectrogram_mask
        )

        # --------------------------------------------------
        # Reparameterization trick
        # z = mean + noise * standard deviation
        # --------------------------------------------------
        noise = torch.randn_like(posterior_mean)
        posterior_std = torch.exp(posterior_log_scale)

        z = (
            posterior_mean
            + noise * posterior_std 
        ) * spectrogram_mask

        return (
            z,
            posterior_mean,
            posterior_log_scale,
            spectrogram_mask
        )

In [ ]:
spec_channels = N_FFT // 2 + 1       # 513
latent_channels = 192
hidden_channels = 192

posterior_encoder = PosteriorEncoder(
    spec_channels=spec_channels,
    latent_channels=latent_channels,
    hidden_channels=hidden_channels
)

spectrogram = batch[2]
spectrogram_lengths = batch[3]

z, posterior_mean, posterior_log_scale, spectrogram_mask = (
    posterior_encoder(
        spectrogram,
        spectrogram_lengths
    )
)

print("Spectrogram:", spectrogram.shape)
print("z:", z.shape)
print("Mean:", posterior_mean.shape)
print("Log-scale:", posterior_log_scale.shape)
print("Mask:", spectrogram_mask.shape)

## 8. Waveform decoder

The decoder behaves like a compact neural vocoder. Four transposed-convolution stages increase time resolution by $8\times8\times2\times2=256$, exactly matching the STFT hop length. Residual blocks refine the waveform after every upsampling stage.

Transposed convolution is used here because it follows the standard VITS/HiFi-GAN design. An alternative is interpolation followed by ordinary convolution, but it is not automatically better and would need to be evaluated experimentally.


In [ ]:
class ResBlock(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size=3,
        dilations=(1, 3, 5)
    ):
        """
        A simple residual block containing three dilated
        convolution stages.
    
        Each stage performs (repeat three times with dilation 1, 3, 5):
    
            input
              ↓
            LeakyReLU
              ↓
            dilated Conv1d
              ↓
            LeakyReLU
              ↓
            normal Conv1d
              ↓
            add original input
        """
        
        super().__init__()

        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.LeakyReLU(0.1),

                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    dilation=dilation,
                    padding=dilation * (kernel_size - 1) // 2
                ),

                nn.LeakyReLU(0.1),

                nn.Conv1d(
                    channels,
                    channels,
                    kernel_size,
                    padding=(kernel_size - 1) // 2
                )
            )
            for dilation in dilations
        ])

    def forward(self, x):
        for layer in self.layers:
            x = x + layer(x)

        return x

In [ ]:
class Decoder(nn.Module):
    """
    Convert a latent representation into a waveform.

    Input:
        [batch, latent_channels, latent_time]

    Output:
        [batch, 1, latent_time * 256]
    """

    def __init__(
        self,
        latent_channels=192,
        initial_channels=512
    ):
        super().__init__()

        # --------------------------------------------------
        # Input projection
        # [B, 192, T] -> [B, 512, T]
        # --------------------------------------------------

        self.input_projection = nn.Conv1d(
            latent_channels,
            initial_channels,
            kernel_size=7,
            padding=3
        )

        # --------------------------------------------------
        # Upsampling stage 1
        # Time: T -> T × 8
        # Channels: 512 -> 256
        # --------------------------------------------------

        self.upsample1 = nn.ConvTranspose1d(
            in_channels=512,
            out_channels=256,
            kernel_size=16,
            stride=8,
            padding=4
        )

        self.resblock1 = ResBlock(
            channels=256,
            kernel_size=3,
            dilations=(1, 3, 5)
        )

        # --------------------------------------------------
        # Upsampling stage 2
        # Time: T × 8 -> T × 64
        # Channels: 256 -> 128
        # --------------------------------------------------

        self.upsample2 = nn.ConvTranspose1d(
            in_channels=256,
            out_channels=128,
            kernel_size=16,
            stride=8,
            padding=4
        )

        self.resblock2 = ResBlock(
            channels=128,
            kernel_size=3,
            dilations=(1, 3, 5)
        )

        # --------------------------------------------------
        # Upsampling stage 3
        # Time: T × 64 -> T × 128
        # Channels: 128 -> 64
        # --------------------------------------------------

        self.upsample3 = nn.ConvTranspose1d(
            in_channels=128,
            out_channels=64,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.resblock3 = ResBlock(
            channels=64,
            kernel_size=3,
            dilations=(1, 3, 5)
        )

        # --------------------------------------------------
        # Upsampling stage 4
        # Time: T × 128 -> T × 256
        # Channels: 64 -> 32
        # --------------------------------------------------

        self.upsample4 = nn.ConvTranspose1d(
            in_channels=64,
            out_channels=32,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.resblock4 = ResBlock(
            channels=32,
            kernel_size=3,
            dilations=(1, 3, 5)
        )

        # --------------------------------------------------
        # Output waveform
        # [B, 32, T × 256] -> [B, 1, T × 256]
        # --------------------------------------------------

        self.output_projection = nn.Conv1d(
            in_channels=32,
            out_channels=1,
            kernel_size=7,
            padding=3
        )

    def forward(self, z):
        # Input projection
        x = self.input_projection(z)
    
        # Upsampling stage 1
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.upsample1(x)
        x = self.resblock1(x)

        # Upsampling stage 2
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.upsample2(x)
        x = self.resblock2(x)
        
        # Upsampling stage 3
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.upsample3(x)
        x = self.resblock3(x)
        
        # Upsampling stage 4
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.upsample4(x)
        x = self.resblock4(x)

        # Generate waveform
        x = F.leaky_relu(x, negative_slope=0.1)
        x = self.output_projection(x)
        
        # Restrict waveform values to [-1, 1]
        waveform = torch.tanh(x)
        return waveform


### Random training segments

The encoders process the **complete** text and spectrogram because alignment needs the whole utterance. Only the latent passed to the decoder is cropped. Decoding 32 frames instead of the complete utterance greatly reduces waveform memory use while keeping the encoder context intact.

Because each frame represents 256 samples, a 32-frame latent segment produces exactly 8,192 waveform samples.


In [ ]:
def slice_segments_at(x, start_indices, segment_size):
    """Slice one segment from every item in a [B, C, T] tensor."""
    segments = []

    for batch_index in range(x.size(0)):
        start = int(start_indices[batch_index].item())
        segment = x[
            batch_index : batch_index + 1,
            :,
            start : start + segment_size,
        ]

        # Padding is only needed when an utterance is shorter than the segment.
        missing = segment_size - segment.size(-1)
        if missing > 0:
            segment = F.pad(segment, (0, missing))

        segments.append(segment)

    return torch.cat(segments, dim=0)


def random_segments(x, lengths, segment_size):
    """Select one valid random start position for every batch item."""
    max_start = (lengths - segment_size).clamp(min=0)
    random_numbers = torch.rand(max_start.shape, device=x.device)
    start_indices = (random_numbers * (max_start + 1).float()).long()

    segments = slice_segments_at(x, start_indices, segment_size)
    return segments, start_indices


z_segment, start_frames = random_segments(
    z,
    spectrogram_lengths,
    SEGMENT_FRAMES,
)

print("Full latent shape:   ", z.shape)
print("Latent segment shape:", z_segment.shape)
print("Start-frame shape:   ", start_frames.shape)


In [ ]:
decoder = Decoder(
    latent_channels=192,
    initial_channels=512,
)

generated_waveform = decoder(z_segment)

print("Latent segment:   ", z_segment.shape)
print("Generated waveform:", generated_waveform.shape)


## 9. Normalizing flow

The posterior encoder and text encoder initially describe different latent distributions. Reversible coupling layers transform the posterior latent into the text-prior space during training and apply the inverse transformation during inference.

This compact version uses additive coupling. Splitting and flipping the channels allows successive layers to update every channel while keeping the transformation invertible.


In [ ]:
class ChannelLayerNorm(nn.Module):
    """LayerNorm for tensors with shape [batch, channels, time]."""

    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        # [B, C, T] -> [B, T, C]
        x = x.transpose(1, 2)

        x = self.norm(x)

        # [B, T, C] -> [B, C, T]
        return x.transpose(1, 2)

class ResidualConvLayer(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size,
        dilation,
        dropout=0.0
    ):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.norm = ChannelLayerNorm(channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        residual = x

        x = self.conv(x * mask)
        x = self.norm(x)
        x = F.relu(x)
        x = self.dropout(x)

        # Residual connection
        return (residual + x) * mask

class CouplingNetwork(nn.Module):
    def __init__(
        self,
        input_channels,
        hidden_channels,
        kernel_size=5,
        dilation_rate=1,
        n_layers=4,
        gin_channels=0
    ):
        super().__init__()

        self.input_projection = nn.Conv1d(
            input_channels,
            hidden_channels,
            kernel_size=1
        )

        self.residual_layers = nn.ModuleList([
            ResidualConvLayer(
                channels=hidden_channels,
                kernel_size=kernel_size,
                dilation=dilation_rate ** layer_index
            )
            for layer_index in range(n_layers)
        ])

        self.output_projection = nn.Conv1d(
            hidden_channels,
            input_channels,
            kernel_size=1
        )

        # Speaker embedding projection
        if gin_channels > 0:
            self.condition_projection = nn.Conv1d(
                gin_channels,
                hidden_channels,
                kernel_size=1
            )
        else:
            self.condition_projection = None

        # Make the coupling layer start as an identity function
        nn.init.zeros_(self.output_projection.weight)
        nn.init.zeros_(self.output_projection.bias)

    def forward(self, x, mask, g=None):
        x = self.input_projection(x * mask)

        if g is not None:
            if self.condition_projection is None:
                raise ValueError(
                    "gin_channels must be greater than 0 when using g."
                )

            # g is normally [B, gin_channels, 1]
            # It broadcasts across the time dimension
            x = x + self.condition_projection(g)

        x = x * mask

        for layer in self.residual_layers:
            x = layer(x, mask)

        shift = self.output_projection(x)

        return shift * mask

class ResidualCouplingLayer(nn.Module):
    def __init__(
        self,
        channels,
        hidden_channels,
        kernel_size=5,
        dilation_rate=1,
        n_layers=4,
        gin_channels=0
    ):
        super().__init__()

        if channels % 2 != 0:
            raise ValueError("channels must be an even number.")

        half_channels = channels // 2

        self.coupling_network = CouplingNetwork(
            input_channels=half_channels,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size,
            dilation_rate=dilation_rate,
            n_layers=n_layers,
            gin_channels=gin_channels
        )

    def forward(self, x, mask, g=None, reverse=False):
        # Split channels into two equal parts
        x_keep, x_change = torch.chunk(x, chunks=2, dim=1)

        # Predict the shift using only x_keep
        shift = self.coupling_network(x_keep, mask, g)

        if reverse:
            # Inverse transformation
            x_change = x_change - shift

            x = torch.cat([x_keep, x_change], dim=1)
            return x * mask

        # Forward transformation
        x_change = x_change + shift

        x = torch.cat([x_keep, x_change], dim=1)
        x = x * mask

        # An additive coupling layer has log determinant = 0
        log_determinant = torch.zeros(
            x.size(0),
            device=x.device,
            dtype=x.dtype
        )

        return x, log_determinant

class ResidualCouplingBlock(nn.Module):
    def __init__(
        self,
        channels,
        hidden_channels,
        kernel_size=5,
        dilation_rate=1,
        n_layers=4,
        n_flows=4,
        gin_channels=0
    ):
        super().__init__()

        self.coupling_layers = nn.ModuleList([
            ResidualCouplingLayer(
                channels=channels,
                hidden_channels=hidden_channels,
                kernel_size=kernel_size,
                dilation_rate=dilation_rate,
                n_layers=n_layers,
                gin_channels=gin_channels
            )
            for _ in range(n_flows)
        ])

    def forward(self, x, mask, g=None, reverse=False):
        if not reverse:
            for coupling_layer in self.coupling_layers:
                x, _ = coupling_layer(
                    x,
                    mask,
                    g=g,
                    reverse=False
                )

                # Replace modules.Flip()
                x = torch.flip(x, dims=[1])

        else:
            # Undo operations in the opposite order
            for coupling_layer in reversed(self.coupling_layers):
                x = torch.flip(x, dims=[1])

                x = coupling_layer(
                    x,
                    mask,
                    g=g,
                    reverse=True
                )

        return x

## 10. Duration predictor

The duration predictor estimates the log number of spectrogram frames assigned to each text token. Its input is detached so the duration objective trains the predictor without directly changing the text encoder.

During training, MAS supplies target durations. During inference, the predicted durations determine the output speech length.


In [ ]:
class DurationConvBlock(nn.Module):
    def __init__(
        self,
        input_channels,
        output_channels,
        kernel_size,
        dropout
    ):
        super().__init__()

        self.conv = nn.Conv1d(
            input_channels,
            output_channels,
            kernel_size,
            padding=kernel_size // 2
        )

        self.norm = ChannelLayerNorm(output_channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.conv(x * mask)
        x = F.relu(x)
        x = self.norm(x)
        x = self.dropout(x)

        return x * mask

class DurationPredictor(nn.Module):
    def __init__(
        self,
        input_channels,
        hidden_channels=256,
        kernel_size=3,
        dropout=0.5,
        gin_channels=0
    ):
        super().__init__()

        self.conv_block_1 = DurationConvBlock(
            input_channels,
            hidden_channels,
            kernel_size,
            dropout
        )

        self.conv_block_2 = DurationConvBlock(
            hidden_channels,
            hidden_channels,
            kernel_size,
            dropout
        )

        # Produce one duration value for each text token
        self.output_projection = nn.Conv1d(
            hidden_channels,
            out_channels=1,
            kernel_size=1
        )

        if gin_channels > 0:
            self.condition_projection = nn.Conv1d(
                gin_channels,
                input_channels,
                kernel_size=1
            )
        else:
            self.condition_projection = None

    def forward(self, x, mask, g=None):
        # Prevent duration loss from updating the text encoder
        x = x.detach()

        if g is not None:
            if self.condition_projection is None:
                raise ValueError(
                    "gin_channels must be greater than 0 when using g."
                )

            g = g.detach()
            x = x + self.condition_projection(g)

        x = self.conv_block_1(x, mask)
        x = self.conv_block_2(x, mask)

        log_duration = self.output_projection(x)

        return log_duration * mask

## 11. Monotonic Alignment Search (MAS)

Text tokens and spectrogram frames have different lengths, so the model needs a monotonic path connecting them. MAS finds the highest-scoring path subject to two constraints: the alignment moves forward in time and every audio frame is assigned to one text token.

The alignment is a discrete search, so it is computed under `torch.no_grad()`. This notebook uses a readable NumPy/Python implementation; an optimized Cython or CUDA version would be faster for large-scale training.


In [ ]:
def maximum_path_each(path, value, t_y, t_x, max_neg_val=-1e9):
    index = t_x - 1
    for y in range(t_y):
        for x in range(max(0, t_x + y - t_y), min(t_x, y + 1)):
            if x == y:
                v_cur = max_neg_val
            else:
                v_cur = value[y-1, x] if y > 0 else max_neg_val
            if x == 0:
                if y == 0:
                    v_prev = 0.
                else:
                    v_prev = max_neg_val
            else:
                v_prev = value[y-1, x-1] if y > 0 and x > 0 else max_neg_val
            value[y, x] += max(v_prev, v_cur)

    for y in range(t_y - 1, -1, -1):
        path[y, index] = 1
        if index != 0 and (index == y or value[y-1, index] < value[y-1, index-1]):
            index = index - 1

def maximum_path_c(paths, values, t_ys, t_xs):
    b = paths.shape[0]
    for i in range(b):
        maximum_path_each(paths[i], values[i], t_ys[i], t_xs[i])

def maximum_path(neg_cent, mask):
    """ Pure Python version.
    neg_cent: [b, t_t, t_s]
    mask: [b, t_t, t_s]
    """
    device = neg_cent.device
    dtype = neg_cent.dtype
    neg_cent = neg_cent.data.cpu().numpy().astype(np.float32)
    path = np.zeros(neg_cent.shape, dtype=np.int32)

    t_t_max = mask.sum(1)[:, 0].data.cpu().numpy().astype(np.int32)
    t_s_max = mask.sum(2)[:, 0].data.cpu().numpy().astype(np.int32)
    maximum_path_c(path, neg_cent, t_t_max, t_s_max)
    return torch.from_numpy(path).to(device=device, dtype=dtype)

In [ ]:
def expand_with_alignment(values, attention):
    """
    Expand [B, C, T_text] values into [B, C, T_spec].

    attention has shape [B, 1, T_spec, T_text].
    """
    return torch.bmm(
        attention.squeeze(1),
        values.transpose(1, 2),
    ).transpose(1, 2)


def make_inference_path(durations, attention_mask):
    """Convert integer token durations into a hard attention path."""
    end_frames = torch.cumsum(durations, dim=-1)
    start_frames = F.pad(end_frames[..., :-1], (1, 0))

    output_frames = attention_mask.size(2)
    frame_ids = torch.arange(
        output_frames,
        device=durations.device,
    ).view(1, 1, output_frames, 1)

    path = (
        (frame_ids >= start_frames.unsqueeze(2))
        & (frame_ids < end_frames.unsqueeze(2))
    )

    return path.to(attention_mask.dtype) * attention_mask


def gaussian_alignment_score(z, prior_mean, prior_log_scale):
    """
    Compute log p(z_frame | text_token) for every frame/token pair.

    Returns [B, T_spec, T_text].
    """
    inverse_variance = torch.exp(-2.0 * prior_log_scale)

    constant = torch.sum(
        -0.5 * math.log(2.0 * math.pi) - prior_log_scale,
        dim=1,
        keepdim=True,
    )

    z_squared = torch.matmul(
        -0.5 * z.pow(2).transpose(1, 2),
        inverse_variance,
    )

    cross_term = torch.matmul(
        z.transpose(1, 2),
        prior_mean * inverse_variance,
    )

    mean_squared = torch.sum(
        -0.5 * prior_mean.pow(2) * inverse_variance,
        dim=1,
        keepdim=True,
    )

    return constant + z_squared + cross_term + mean_squared


## 12. Complete VITS-style model

The training forward pass encodes text and target audio, maps the posterior latent through the flow, finds the MAS alignment, predicts token durations, and decodes one random latent segment.

Inference uses text only: predicted durations create an alignment, a latent sample is drawn from the expanded prior, the flow is inverted, and the decoder generates the waveform.


In [ ]:
class VITSModel(nn.Module):
    """A small single-speaker VITS model using the blocks in the notebook."""

    def __init__(
        self,
        vocab_size,
        spec_channels,
        padding_id=0,
        latent_channels=192,
        hidden_channels=192,
        feedforward_channels=768,
        decoder_channels=512,
        segment_samples=8192,
        hop_length=256,
    ):
        super().__init__()

        if segment_samples % hop_length != 0:
            raise ValueError("segment_samples must be divisible by hop_length.")

        self.hop_length = hop_length
        self.segment_samples = segment_samples
        self.segment_frames = segment_samples // hop_length

        self.text_encoder = TextEncoder(
            vocab_size=vocab_size,
            hidden_channels=hidden_channels,
            latent_channels=latent_channels,
            feedforward_channels=feedforward_channels,
            padding_id=padding_id,
        )

        self.posterior_encoder = PosteriorEncoder(
            spec_channels=spec_channels,
            latent_channels=latent_channels,
            hidden_channels=hidden_channels,
        )

        self.flow = ResidualCouplingBlock(
            channels=latent_channels,
            hidden_channels=hidden_channels,
        )

        self.duration_predictor = DurationPredictor(
            input_channels=hidden_channels,
        )

        self.decoder = Decoder(
            latent_channels=latent_channels,
            initial_channels=decoder_channels,
        )

    def forward(
        self,
        text_tokens,
        text_lengths,
        spectrogram,
        spectrogram_lengths,
    ):
        (
            text_features,
            prior_mean,
            prior_log_scale,
            text_mask,
        ) = self.text_encoder(text_tokens, text_lengths)

        (
            z,
            posterior_mean,
            posterior_log_scale,
            spectrogram_mask,
        ) = self.posterior_encoder(spectrogram, spectrogram_lengths)

        # Posterior latent space -> text-prior latent space
        z_prior = self.flow(z, spectrogram_mask)

        # MAS is a hard, non-differentiable alignment search.
        with torch.no_grad():
            score = gaussian_alignment_score(
                z_prior.float(),
                prior_mean.float(),
                prior_log_scale.float(),
            )

            alignment_mask = (
                spectrogram_mask.float().transpose(1, 2)
                * text_mask.float()
            )

            attention = maximum_path(
                score,
                alignment_mask,
            ).unsqueeze(1)

            attention = attention.to(text_features.dtype)

        # Number of spectrogram frames assigned to each text token
        target_duration = attention.sum(dim=2)
        target_log_duration = (
            torch.log(target_duration + 1e-6) * text_mask
        )

        predicted_log_duration = self.duration_predictor(
            text_features,
            text_mask,
        )

        # Expand the text prior to spectrogram time.
        expanded_prior_mean = expand_with_alignment(
            prior_mean,
            attention,
        )

        expanded_prior_log_scale = expand_with_alignment(
            prior_log_scale,
            attention,
        )

        # Decode only a short random segment to reduce memory usage.
        z_segment, start_frames = random_segments(
            z,
            spectrogram_lengths,
            self.segment_frames,
        )

        generated_audio = self.decoder(z_segment)

        return {
            "generated_audio": generated_audio,
            "start_frames": start_frames,
            "attention": attention,
            "text_mask": text_mask,
            "spectrogram_mask": spectrogram_mask,
            "predicted_log_duration": predicted_log_duration,
            "target_log_duration": target_log_duration,
            "z_prior": z_prior,
            "posterior_mean": posterior_mean,
            "posterior_log_scale": posterior_log_scale,
            "prior_mean": expanded_prior_mean,
            "prior_log_scale": expanded_prior_log_scale,
        }

    @torch.inference_mode()
    def infer(
        self,
        text_tokens,
        text_lengths,
        noise_scale=1.0,
        length_scale=1.0,
        max_frames=None,
        max_duration_per_token=50,
    ):
        (
            text_features,
            prior_mean,
            prior_log_scale,
            text_mask,
        ) = self.text_encoder(text_tokens, text_lengths)

        predicted_log_duration = self.duration_predictor(
            text_features,
            text_mask,
        )

        # The clamp prevents an early, untrained model from producing
        # an extremely long waveform and exhausting GPU memory.
        maximum_log_duration = math.log(max_duration_per_token)
        predicted_log_duration = predicted_log_duration.clamp(
            min=-5.0,
            max=maximum_log_duration,
        )

        durations = torch.ceil(
            torch.exp(predicted_log_duration) * length_scale
        ) * text_mask

        output_lengths = durations.sum(dim=(1, 2)).long().clamp(min=1)

        if max_frames is not None:
            output_lengths = output_lengths.clamp(max=max_frames)

        output_frames = int(output_lengths.max().item())

        spectrogram_mask = create_sequence_mask(
            output_lengths,
            output_frames,
        ).unsqueeze(1).to(text_features.dtype)

        attention_mask = (
            spectrogram_mask.unsqueeze(-1)
            * text_mask.unsqueeze(2)
        )

        attention = make_inference_path(
            durations,
            attention_mask,
        )

        expanded_prior_mean = expand_with_alignment(
            prior_mean,
            attention,
        )

        expanded_prior_log_scale = expand_with_alignment(
            prior_log_scale,
            attention,
        )

        noise = torch.randn_like(expanded_prior_mean)
        z_prior = (
            expanded_prior_mean
            + noise
            * torch.exp(expanded_prior_log_scale)
            * noise_scale
        )

        z = self.flow(
            z_prior,
            spectrogram_mask,
            reverse=True,
        )

        generated_audio = self.decoder(z * spectrogram_mask)
        audio_lengths = output_lengths * self.hop_length

        return {
            "generated_audio": generated_audio,
            "audio_lengths": audio_lengths,
            "attention": attention,
        }

## 13. Training objectives

The non-adversarial objective combines four terms:

- **Mel loss:** compares log-mel spectrograms and strongly guides perceptual content.
- **Multi-resolution STFT loss:** compares spectral structure at several time-frequency resolutions.
- **Duration loss:** matches predicted and MAS-derived log durations.
- **KL loss:** encourages the posterior latent, after the flow, to match the text-conditioned prior.

The mel and STFT calculations are forced to `float32` for numerical stability even when mixed-precision training is enabled.


In [ ]:
class MultiResolutionSTFTLoss(nn.Module):
    """Compare generated and real audio at several STFT resolutions."""

    def __init__(
        self,
        resolutions=(
            (512, 128, 512),
            (1024, 256, 1024),
            (2048, 512, 2048),
        ),
        epsilon=1e-7,
    ):
        super().__init__()
        self.resolutions = resolutions
        self.epsilon = epsilon

        for index, (_, _, window_length) in enumerate(resolutions):
            self.register_buffer(
                f"window_{index}",
                torch.hann_window(window_length),
                persistent=False,
            )

    def _magnitude(self, waveform, resolution_index):
        fft_size, hop_length, window_length = self.resolutions[
            resolution_index
        ]

        window = getattr(self, f"window_{resolution_index}")

        stft = torch.stft(
            waveform,
            n_fft=fft_size,
            hop_length=hop_length,
            win_length=window_length,
            window=window,
            return_complex=True,
        )

        return stft.abs().clamp(min=self.epsilon)

    def forward(self, generated, target):
        spectral_convergence = generated.new_tensor(0.0)
        log_magnitude_loss = generated.new_tensor(0.0)

        for index in range(len(self.resolutions)):
            generated_magnitude = self._magnitude(generated, index)
            target_magnitude = self._magnitude(target, index)

            difference_norm = torch.linalg.vector_norm(
                target_magnitude - generated_magnitude,
                dim=(-2, -1),
            )

            target_norm = torch.linalg.vector_norm(
                target_magnitude,
                dim=(-2, -1),
            ).clamp(min=self.epsilon)

            spectral_convergence += (
                difference_norm / target_norm
            ).mean()

            log_magnitude_loss += F.l1_loss(
                torch.log(generated_magnitude),
                torch.log(target_magnitude),
            )

        number_of_resolutions = len(self.resolutions)

        return (
            spectral_convergence / number_of_resolutions,
            log_magnitude_loss / number_of_resolutions,
        )


class VITSLoss(nn.Module):
    """Non-adversarial VITS loss: mel + STFT + duration + KL."""

    def __init__(
        self,
        sample_rate=22050,
        n_fft=1024,
        hop_length=256,
        win_length=1024,
        n_mels=80,
        mel_weight=10.0,
        stft_weight=1.0,
        duration_weight=1.0,
        kl_weight=1.0,
        wave_weight=10
        
    ):
        super().__init__()

        self.mel_weight = mel_weight
        self.stft_weight = stft_weight
        self.duration_weight = duration_weight
        self.kl_weight = kl_weight
        self.wave_weight = wave_weight

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            n_mels=n_mels,
            power=1.0,
        )

        self.stft_loss = MultiResolutionSTFTLoss()

    @staticmethod
    def _log_mel(mel):
        return torch.log(mel.clamp(min=1e-5))

    @staticmethod
    def _duration_loss(outputs):
        difference = (
            outputs["predicted_log_duration"].float()
            - outputs["target_log_duration"].float()
        ).pow(2)

        mask = outputs["text_mask"].float()
        return (difference * mask).sum() / mask.sum().clamp(min=1.0)

    @staticmethod
    def _kl_loss(outputs):
        z_prior = outputs["z_prior"].float()
        posterior_log_scale = outputs["posterior_log_scale"].float()
        prior_mean = outputs["prior_mean"].float()
        prior_log_scale = outputs["prior_log_scale"].float()
        mask = outputs["spectrogram_mask"].float()

        kl = prior_log_scale - posterior_log_scale - 0.5
        kl = kl + 0.5 * (
            (z_prior - prior_mean).pow(2)
            * torch.exp(-2.0 * prior_log_scale)
        )

        return (kl * mask).sum() / mask.sum().clamp(min=1.0)

    def forward(self, outputs, target_audio):
        generated_audio = outputs["generated_audio"].float().squeeze(1)
        target_audio = target_audio.float().squeeze(1)

        generated_mel = self._log_mel(
            self.mel_transform(generated_audio)
        )

        target_mel = self._log_mel(
            self.mel_transform(target_audio)
        )

        mel_loss = F.l1_loss(generated_mel, target_mel)

        spectral_convergence, log_magnitude_loss = self.stft_loss(
            generated_audio,
            target_audio,
        )

        stft_loss = spectral_convergence + log_magnitude_loss
        duration_loss = self._duration_loss(outputs)
        kl_loss = self._kl_loss(outputs)

        waveform_loss = F.l1_loss(
            generated_audio,
            target_audio,
        )

        # using gan weight to control the loss 
        # These value might be useful when createing gan loss:
        # [target_audio, generated_audio] or [target_mel, generated_mel]
        # just pick one value set to implement the loss
        
        
        # Add you GAN Loss Here
        gan_weight = 10 
        gan_loss = 0
        gan_loss = gan_loss * gan_weight

        device = generated_audio.device
        gan_loss = torch.tensor(gan_loss).to(device)
        
        total_loss = (
            self.mel_weight * mel_loss
            + self.stft_weight * stft_loss
            + self.duration_weight * duration_loss
            + self.kl_weight * kl_loss
            + self.wave_weight * waveform_loss
            + gan_loss
        )

        return {
            "total": total_loss,
            "mel": mel_loss,
            "stft": stft_loss,
            "duration": duration_loss,
            "kl": kl_loss,
            "wave": waveform_loss,
            "gan": gan_loss
        }


In [ ]:
# Create a discriminator used for GAN loss

class Discriminator(nn.Module):
    def __init__(
        self,
    ):
        super().__init__()
        

## 14. Trainer: two GPUs and mixed precision

For Kaggle's **T4 ×2** accelerator, the trainer wraps the model with `nn.DataParallel`. The global batch is copied to GPU 0, split across both GPUs for the model forward/backward pass, and gathered on GPU 0 for loss calculation. Sampling uses the underlying base model because custom methods such as `infer()` are not exposed directly by the wrapper.

Checkpoints also save the base model's state dictionary. This avoids a `module.` prefix and keeps checkpoints compatible with both single-GPU and multi-GPU runs.

Mixed precision uses the current API:

```python
torch.amp.GradScaler("cuda", enabled=use_amp)
```

This replaces the deprecated `torch.cuda.amp.GradScaler(...)` call.


In [ ]:
class VITSTrainer:
    def __init__(
        self,
        model,
        loss_function,
        train_loader,
        learning_rate=1e-4,
        checkpoint_dir="./model_weight",
        sample_dir="./generated_samples",
        device=None,
        use_amp=True,
        use_multi_gpu=True,
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        self.device = torch.device(device)
        self.train_loader = train_loader

        # DataParallel is convenient in a Kaggle notebook because it uses one
        # Python process, replicates the model, and splits each batch over all
        # visible GPUs. GPU 0 gathers the outputs and calculates the loss.
        self.gpu_ids = list(range(torch.cuda.device_count()))
        self.use_multi_gpu = (
            use_multi_gpu
            and self.device.type == "cuda"
            and len(self.gpu_ids) > 1
        )

        self.model = model.to(self.device)
        if self.use_multi_gpu:
            self.model = nn.DataParallel(
                self.model,
                device_ids=self.gpu_ids,
                output_device=self.gpu_ids[0],
            )

        # Custom methods and attributes, such as infer() and hop_length, live
        # on the underlying model rather than on the DataParallel wrapper.
        self.base_model = (
            self.model.module
            if isinstance(self.model, nn.DataParallel)
            else self.model
        )

        self.loss_function = loss_function.to(self.device)

        if self.use_multi_gpu:
            gpu_names = [
                torch.cuda.get_device_name(gpu_id)
                for gpu_id in self.gpu_ids
            ]
            print(
                f"Training with {len(self.gpu_ids)} GPUs via "
                f"nn.DataParallel: {gpu_names}"
            )
        else:
            print(f"Training on one device: {self.device}")

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=learning_rate,
            betas=(0.8, 0.99),
            eps=1e-9,
            weight_decay=0.01,
        )

        # AMP may skip optimizer.step while the training loop still advances.
        # Keep attempted iterations and actual updates as distinct evidence.
        self.optimizer_update_stats = {"updates": 0}
        update_stats = self.optimizer_update_stats

        def record_optimizer_update(optimizer, arguments, keywords):
            if update_stats["updates"] is not None:
                update_stats["updates"] += 1

        self._optimizer_update_hook = self.optimizer.register_step_post_hook(
            record_optimizer_update
        )

        self.use_amp = use_amp and self.device.type == "cuda"
        self.scaler = torch.amp.GradScaler(
            "cuda",
            enabled=self.use_amp,
        )
        self.step = 0

        self.checkpoint_dir = Path(checkpoint_dir)
        self.sample_dir = Path(sample_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.sample_dir.mkdir(parents=True, exist_ok=True)

    def _move_batch_to_device(self, batch):
        return tuple(
            item.to(self.device, non_blocking=True)
            for item in batch
        )

    def save_checkpoint(self):
        checkpoint = {
            "step": self.step,
            "optimizer_updates": self.optimizer_update_stats["updates"],
            "model": self.base_model.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "scaler": self.scaler.state_dict(),
        }

        path = self.checkpoint_dir / f"vits_step_{self.step}.pt"
        torch.save(checkpoint, path)
        print(f"Saved checkpoint: {path}")

    def load_checkpoint(self, path):
        checkpoint = torch.load(path, map_location=self.device)
        self.base_model.load_state_dict(checkpoint["model"])
        self.optimizer.load_state_dict(checkpoint["optimizer"])
        self.scaler.load_state_dict(checkpoint["scaler"])
        self.step = checkpoint["step"]
        # Missing counts in older checkpoints are unknown, not zero.
        self.optimizer_update_stats["updates"] = checkpoint.get("optimizer_updates")

        print(f"Resumed from step {self.step}")

    @torch.inference_mode()
    def save_samples(self, batch, number_of_samples=2):
        was_training = self.model.training
        self.model.eval()

        text_tokens, text_lengths = batch[0], batch[1]
        text_tokens = text_tokens[:number_of_samples]
        text_lengths = text_lengths[:number_of_samples]

        result = self.base_model.infer(
            text_tokens,
            text_lengths,
            noise_scale=0.8,
            max_frames=1000,
        )

        for index in range(text_tokens.size(0)):
            audio_length = int(result["audio_lengths"][index].item())
            waveform = result["generated_audio"][
                index,
                :,
                :audio_length,
            ].float().cpu().clamp(-1.0, 1.0)

            path = self.sample_dir / (
                f"step_{self.step}_sample_{index + 1}.wav"
            )

            torchaudio.save(str(path), waveform, sample_rate=22050)

        self.model.train(was_training)

    def train(
        self,
        total_steps,
        save_every=2000,
        log_every=20,
        max_grad_norm=1.0,
    ):
        self.model.train()

        progress = tqdm(
            total=total_steps,
            initial=self.step,
            desc="Training",
        )

        while self.step < total_steps:
            for batch in self.train_loader:
                if self.step >= total_steps:
                    break

                batch = self._move_batch_to_device(batch)

                (
                    text_tokens,
                    text_lengths,
                    spectrogram,
                    spectrogram_lengths,
                    waveform,
                    _,
                ) = batch

                self.optimizer.zero_grad(set_to_none=True)

                with torch.autocast(
                    device_type=self.device.type,
                    dtype=torch.float16,
                    enabled=self.use_amp,
                ):
                    outputs = self.model(
                        text_tokens,
                        text_lengths,
                        spectrogram,
                        spectrogram_lengths,
                    )

                target_start_samples = (
                    outputs["start_frames"] * self.base_model.hop_length
                )

                target_audio = slice_segments_at(
                    waveform,
                    target_start_samples,
                    self.base_model.segment_samples,
                )

                # STFT and mel losses are calculated in float32.
                with torch.autocast(
                    device_type=self.device.type,
                    enabled=False,
                ):
                    losses = self.loss_function(outputs, target_audio)

                total_loss = losses["total"]

                if not torch.isfinite(total_loss):
                    raise FloatingPointError(
                        f"Non-finite loss at step {self.step + 1}."
                    )

                self.scaler.scale(total_loss).backward()
                self.scaler.unscale_(self.optimizer)

                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    max_grad_norm,
                )

                self.scaler.step(self.optimizer)
                self.scaler.update()

                self.step += 1
                progress.update(1)

                if self.step % log_every == 0:
                    progress.set_postfix({
                        name: f"{value.detach().item():.3f}"
                        for name, value in losses.items()
                    })

                if self.step % save_every == 0:
                    self.save_checkpoint()
                    self.save_samples(batch)

        progress.close()

    def overfit_train(
        self,
        total_steps,
        save_every=2000,
        log_every=20,
        max_grad_norm=1.0,
        fixed_batch=None,
        evaluator=None,
        eval_every=500,
        run_dir=None,
    ):
        if min(log_every, save_every, eval_every) <= 0:
            raise ValueError("Logging, save and evaluation intervals must be positive.")
        if total_steps <= self.step:
            raise ValueError("total_steps must exceed the current optimizer step.")
        if evaluator is not None and self.step != 0:
            raise ValueError("Comparison runs start from step zero; use a fresh trainer.")
        self.model.train()
        self.training_history = []

        progress = tqdm(
            total=total_steps,
            initial=self.step,
            desc="Training",
        )

        if fixed_batch is None:
            fixed_batch = next(iter(self.train_loader))
        if evaluator is not None:
            if any(not torch.equal(a.detach().cpu(), b) for a, b in zip(fixed_batch, evaluator.batch)):
                raise ValueError("Training batch differs from the evaluator's frozen batch.")
            evaluator.evaluate(self.base_model, self.step)
        batch = self._move_batch_to_device(fixed_batch)
        
        while self.step < total_steps:
            
            (
                text_tokens,
                text_lengths,
                spectrogram,
                spectrogram_lengths,
                waveform,
                _,
            ) = batch

            self.optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=self.device.type,
                dtype=torch.float16,
                enabled=self.use_amp,
            ):
                outputs = self.model(
                    text_tokens,
                    text_lengths,
                    spectrogram,
                    spectrogram_lengths,
                )

            target_start_samples = (
                outputs["start_frames"] * self.base_model.hop_length
            )

            target_audio = slice_segments_at(
                waveform,
                target_start_samples,
                self.base_model.segment_samples,
            )

            # STFT and mel losses are calculated in float32.
            with torch.autocast(
                device_type=self.device.type,
                enabled=False,
            ):
                losses = self.loss_function(outputs, target_audio)

            total_loss = losses["total"]

            if not torch.isfinite(total_loss):
                raise FloatingPointError(
                    f"Non-finite loss at step {self.step + 1}."
                )

            self.scaler.scale(total_loss).backward()
            self.scaler.unscale_(self.optimizer)

            torch.nn.utils.clip_grad_norm_(
                self.model.parameters(),
                max_grad_norm,
            )

            self.scaler.step(self.optimizer)
            self.scaler.update()

            self.step += 1
            progress.update(1)

            if self.step % log_every == 0 or self.step == total_steps:
                progress.set_postfix({
                    name: f"{value.detach().item():.3f}"
                    for name, value in losses.items()
                })

                # Training outputs precede this update and use random crops.
                # Keep these losses separate from freshly recomputed evaluation.
                self.training_history.append({
                    "step": self.step,
                    "optimizer_updates": self.optimizer_update_stats["updates"],
                    **{name: float(value.detach().item()) for name, value in losses.items()},
                })
                if run_dir is not None:
                    (Path(run_dir) / "training_history.json").write_text(
                        json.dumps(self.training_history, indent=2, allow_nan=False),
                        encoding="utf-8",
                    )

            if evaluator is not None and (self.step % eval_every == 0 or self.step == total_steps):
                summary = evaluator.evaluate(
                    self.base_model, self.step, save_audio=self.step == total_steps,
                )
                print("\nFixed-crop reconstruction:", summary)

            if self.step % save_every == 0 or self.step == total_steps:
                self.save_checkpoint()
                #self.save_samples(batch)

        progress.close()
        return batch



In [ ]:
# Shared evaluation implementation lives outside this notebook.
import sys
import os
import json
import random
import hashlib
from datetime import datetime, timezone

# Local Jupyter: repository root or notebooks/. Kaggle: set STS_REPO_ROOT
# to a checkout/dataset containing BOTH notebooks/ and sts_eval/.
REPO_ROOT = Path(os.environ.get("STS_REPO_ROOT", ".." if Path.cwd().name == "notebooks" else ".")).resolve()
if not (REPO_ROOT / "sts_eval" / "reconstruction.py").is_file():
    raise FileNotFoundError("Set STS_REPO_ROOT to the STS2026 repository containing sts_eval/.")
sys.path.insert(0, str(REPO_ROOT))

from sts_eval import EvaluationConfig, ReconstructionEvaluator, plot_history
from sts_eval.reconstruction import repository_provenance

TRAIN_SEED = 2026
EVAL_SEED = 1234
TOTAL_STEPS = 2000
EVAL_EVERY = 500
MODEL_VARIANT = "original"
# These exact utterances form the overfit training AND evaluation batch.
# Keep these IDs unchanged when comparing the new model.
FIXED_IDS = [f"LJ001-{i:04d}" for i in range(1, 9)]
random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

indexed_metadata = metadata_df.set_index("file_name", drop=False)
if not indexed_metadata.index.is_unique:
    raise ValueError("Duplicate file IDs in metadata.")
fixed_metadata = indexed_metadata.loc[FIXED_IDS].reset_index(drop=True)
fixed_dataset = LJSpeechDataset(fixed_metadata, WAV_DIR)
fixed_batch = collate_tts_batch([fixed_dataset[i] for i in range(len(fixed_dataset))])

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = Path(os.environ.get("STS_RUN_ROOT", str(REPO_ROOT / "output" / "eval"))) / f"{MODEL_VARIANT}_{run_id}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

evaluation_config = EvaluationConfig(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH, segment_frames=SEGMENT_FRAMES, seed=EVAL_SEED,
)
run_config = {
    "variant": MODEL_VARIANT, "train_seed": TRAIN_SEED,
    "total_steps": TOTAL_STEPS, "eval_every": EVAL_EVERY,
    "sample_ids": FIXED_IDS, "data_dir": str(DATA_DIR),
    "metadata_sha256": hashlib.sha256(METADATA_PATH.read_bytes()).hexdigest(),
    "notebook_sha256": hashlib.sha256((REPO_ROOT / "notebooks" / "vits-light_v1.ipynb").read_bytes()).hexdigest(),
    "eval_source_sha256": hashlib.sha256((REPO_ROOT / "sts_eval" / "reconstruction.py").read_bytes()).hexdigest(),
    "repository": repository_provenance(REPO_ROOT),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
    "torch_cuda": torch.version.cuda,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "scope": "fixed training batch reconstruction; not held-out evaluation",
}
evaluator = ReconstructionEvaluator(
    fixed_batch, FIXED_IDS, RUN_DIR / "evaluation",
    config=evaluation_config, provenance=run_config,
)
print("Run artifacts:", RUN_DIR)


## 15. Build and train the old-model baseline

The previous cell freezes eight LJSpeech IDs and the evaluation protocol before
model construction. This experiment compares posterior reconstruction on the
training batch. It does not establish generalization or text-to-speech quality.

Evaluation runs at step 0 and every 500 iterations through step 2000. This is a
fixed budget, not a claim of convergence. The final cell plots losses and fixed
crop metrics, and plays the exact target/mean/sample waveforms that were scored.
See `docs/EVALUATION.md` and `docs/LOCAL_EVALUATION.md` for the protocol and run instructions.


In [ ]:
model = VITSModel(
    vocab_size=len(SYMBOLS),
    padding_id=PAD_ID,
    spec_channels=N_FFT // 2 + 1,
    latent_channels=192,
    hidden_channels=192,
    feedforward_channels=768,
    decoder_channels=512,
    segment_samples=SEGMENT_SAMPLES,
    hop_length=HOP_LENGTH,
)

# Set duration_weight and kl_weight to 0
# We are not doing text in this stage
loss_function = VITSLoss(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    n_mels=80,
    mel_weight=10.0,
    stft_weight=1.0,
    duration_weight=0.0,
    kl_weight=0.0,
    wave_weight=10.0
)


trainer = VITSTrainer(
    model=model,
    loss_function=loss_function,
    train_loader=train_loader,
    learning_rate=1e-4,
    checkpoint_dir=RUN_DIR / "checkpoints",
    sample_dir=RUN_DIR / "samples",
    use_amp=True,
    use_multi_gpu=False,
)
run_config["model"] = repr(model)
run_config["optimizer"] = {k: v for k, v in trainer.optimizer.param_groups[0].items() if k != "params"}
run_config["loss_weights"] = {name: getattr(loss_function, name) for name in
                              ("mel_weight", "stft_weight", "duration_weight", "kl_weight", "wave_weight")}
run_config["amp"] = trainer.use_amp
run_config["multi_gpu"] = trainer.use_multi_gpu
(RUN_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")


In [ ]:
# Runs training. Do not execute this cell just to check installation.
try:
    t_batch = trainer.overfit_train(
        total_steps=TOTAL_STEPS, save_every=EVAL_EVERY, log_every=50,
        fixed_batch=fixed_batch, evaluator=evaluator,
        eval_every=EVAL_EVERY, run_dir=RUN_DIR,
    )
except BaseException as error:
    (RUN_DIR / "status.json").write_text(json.dumps({
        "status": "failed_or_interrupted", "step": trainer.step,
        "ended_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "error_type": type(error).__name__, "error": str(error),
    }, indent=2), encoding="utf-8")
    raise
else:
    (RUN_DIR / "status.json").write_text(json.dumps({
        "status": "completed", "step": trainer.step,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "completion_time_source": "runtime_clock",
        "actual_optimizer_updates": trainer.optimizer_update_stats["updates"],
        "amp_skipped_updates": trainer.step - trainer.optimizer_update_stats["updates"],
    }, indent=2), encoding="utf-8")

print("Experiment record:", (RUN_DIR / "status.json").read_text(encoding="utf-8"))


In [ ]:
# Inspect metrics and the EXACT final crops used for scoring.
import matplotlib.pyplot as plt
from IPython.display import Audio, display

figure = plot_history(evaluator.history, trainer.training_history, RUN_DIR / "curves.png")
plt.show()
print(pd.DataFrame(evaluator.history).tail(2).to_string(index=False))
audio_dir = evaluator.output_dir / f"step_{trainer.step:06d}"
for name in ("target", "mean", "sample"):
    print(FIXED_IDS[0], name)
    display(Audio(filename=str(audio_dir / f"000_{name}.wav")))
print("All sample IDs, crops, metrics and audio are stored in", RUN_DIR)
